<a href="https://colab.research.google.com/github/network-evolution/ai_engineering_notebooks/blob/main/run_gemma4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 01 Check GPU



In [1]:
!nvidia-smi

Thu Oct 15 14:56:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 02 Install Ollama


In [2]:
! sudo apt update && sudo apt install -y pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,919 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,058 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,548 kB]
Get:13 http://archive.ubuntu.com/

## 03 Run Ollama in the backround


In [3]:
import subprocess
import time

# Start ollama as a background process
command = "nohup ollama serve &"
subprocess.run(command, shell=True)

# Give the server a few seconds to initialize
time.sleep(3)
print("Ollama server started in the background.")

Ollama server started in the background.


## 04 Send Prompt to the Model


In [5]:
import requests
import json

url = "http://localhost:11434/api/generate"
payload = {
    "model": "gemma4:e2b",
    # "model": "llama3.2:1b",
    "prompt": "give one line definitions for CPU, GPU and TPU",
    "stream": False,
    "keep_alive": -1
}

response = requests.post(url, json=payload)
data = json.loads(response.text)

print(data["response"])

**CPU (Central Processing Unit):**
The brain of the computer, responsible for executing general instructions and managing overall system operations.

**GPU (Graphics Processing Unit):**
A specialized processor designed for highly parallel computations, excels at graphics rendering and accelerating parallel tasks like machine learning.

**TPU (Tensor Processing Unit):**
A specialized accelerator optimized exclusively for high-performance machine learning, deep learning, and intensive matrix computations.


## 05 Unload Model from GPU RAM

In [16]:
requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "llama3.2:1b",
        "keep_alive": 0
    }
)

<Response [200]>

In [6]:
!pip install ollama

In [7]:
import ollama

prompt = "Give one-line definitions for CPU, GPU and TPU."

settings = {
    "Greedy    (k=5,  p=0.5, t=0.2)": {"top_k": 5,   "top_p": 0.5,  "temperature": 0},
    "Strict with Temp   (k=5,  p=0.5, t=0.2)": {"top_k": 5,   "top_p": 0.5,  "temperature": 0.2},
    "Strict with Seed   (k=5,  p=0.5, t=0.2)": {"top_k": 5,   "top_p": 0.5,  "temperature": 0.2, "seed": 42},
    "Default   (k=64, p=0.95, t=1.0)": {"top_k": 64,  "top_p": 0.95, "temperature": 1.0},
    "Wide open (k=500, p=1.0, t=1.5)": {"top_k": 500, "top_p": 1.0,  "temperature": 1.5},
}

for name, opts in settings.items():
    print(f"\n=== {name} ===")
    for i in range(3):
        r = ollama.generate(
            model="gemma4:e2b",
            system="Answer directly. No introduction, no closing remarks.",
            prompt=prompt,
            options={**opts, "num_predict": 200},
            think=False,
        )
        print(f"\n--- Run {i+1} ({r['eval_count']} tokens, reason: {r['done_reason']}) ---")
        print(r["response"].strip())


=== Greedy    (k=5,  p=0.5, t=0.2) ===

--- Run 1 (68 tokens, reason: stop) ---
CPU: The central processing unit is the brain of a computer, responsible for general-purpose processing and control.
GPU: The graphics processing unit is specialized for parallel processing, excelling at rendering graphics and complex computations.
TPU: The tensor processing unit is a specialized integrated circuit designed to accelerate machine learning and deep learning workloads.

--- Run 2 (64 tokens, reason: stop) ---
CPU: The central processing unit is the brain of a computer, responsible for executing general-purpose instructions.
GPU: The graphics processing unit is specialized for parallel processing, excelling at rendering graphics and complex computations.
TPU: The tensor processing unit is a specialized integrated circuit designed to accelerate machine learning workloads.

--- Run 3 (64 tokens, reason: stop) ---
CPU: The central processing unit is the brain of a computer, responsible for execut